<a id="top"></a>

# Demo: routing -- classify then branch, and who gets to decide

```yaml
title: "Demo: routing -- classify then branch, and who gets to decide"
keywords: langgraph, routing, conditional edge, classifier, mixed models, per-node model, user-input branching, deterministic, haiku, sonnet, Langfuse, workflow, teacher demo
estimated duration: 26
```

> **Lesson:** L05. Teacher demo -- Demos 1 & 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L05/demos_or_activities.md).
> Builds on [L03's chaining demo](../L03/L0305_lecture.ipynb): same support-ticket domain, same
> `ChatAnthropic` setup. Makes **live** Claude calls -- set `ANTHROPIC_API_KEY` (the graph build and
> diagrams run without a key). Cheap classifier on **Haiku 4.5**, capable branches on
> **Sonnet 4.6**. Clear outputs before committing.

## Contents

- [1. Setup -- clients, tickets, tracing](#1-setup----clients-tickets-tracing)
- [2. The routing state and its nodes](#2-the-routing-state-and-its-nodes)
- [3. The conditional edge: branch on a label in state](#3-the-conditional-edge-branch-on-a-label-in-state)
- [4. Run it -- and prove it's deterministic](#4-run-it----and-prove-its-deterministic)
- [5. Read the trace -- a model per node](#5-read-the-trace-a-forward-pointing-taste-not-a-prerequisite----a-model-per-node)
- [6. Same graph, different decider: user input](#6-same-graph-different-decider-user-input)
- [7. Takeaways](#7-takeaways)

## 1. Setup -- clients, tickets, tracing

The same self-contained setup as Demo 1: two `ChatAnthropic` clients (Haiku + Sonnet), the sample
tickets, and the `trace_callbacks()` helper. (Re-run here so this notebook stands on its own.)

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langchain_anthropic import ChatAnthropic
from langgraph.graph import END, StateGraph

from fluffy_potato_curriculum.common.config import (
    get_settings,
    langfuse_configured,
    require_anthropic_key,
)

SONNET = "claude-sonnet-4-6"  # course anchor -- the capable model for heavy reasoning nodes
HAIKU = "claude-haiku-4-5"  # the cheap, fast model for the light nodes (parse / classify / route)

# A handful of support tickets for the running example (the lesson's chosen domain).
TICKETS = {
    "billing": "I was charged twice for my subscription this month -- please refund the extra charge.",
    "technical": "The export button throws a 500 error every time I click it on the reports page.",
    "general": "Do you offer a student discount, and how would I apply it to my account?",
}

# A short policy the policy-check node holds a drafted reply against.
POLICY = (
    "Refunds: a duplicate charge is always refundable within 60 days. "
    "Never promise a refund for change-of-mind. "
    "Escalate anything mentioning legal action or chargebacks to a human."
)

# Live calls load the key through the config seam (common/config.py) -- same as L01-L02,
# only the *client* is the framework's now (continuing from L03). Build the clients only
# when a key is present so a keyless restart-and-run-all still passes; the run cells below
# are gated on LIVE.
LIVE = bool(get_settings().anthropic_api_key)
if LIVE:
    # Per-node model binding: each node uses its OWN client. A node is an independent call,
    # so a graph can mix a cheap model and a capable one (the mechanism from L03).
    haiku = ChatAnthropic(model=HAIKU, api_key=require_anthropic_key(), max_tokens=256)
    sonnet = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)


def trace_callbacks() -> list[object]:
    """Return LangChain callbacks that route spans to the shared Langfuse, or [].

    When Langfuse isn't configured the workflow runs unchanged -- only the spans are
    absent -- so this never blocks a run. L12 is where the Langfuse seam is TAUGHT in full;
    here we just get a forward-pointing taste, same as L03's demo.
    """
    if not langfuse_configured():
        print("Langfuse not configured -- running without tracing. (L12 covers this in full.)")
        return []
    from langfuse.langchain import CallbackHandler

    return [CallbackHandler()]

[↑ Back to top](#top)

## 2. The routing state and its nodes

A **routing** workflow: an entry `classify` node labels the ticket, then a conditional edge sends
it to one of three specialized branch nodes that converge to `END`.

Per-node models again: `classify` is just producing a label, so it runs on **Haiku**; each branch
writes a real reply, so it runs on **Sonnet**.

> The nodes call `await haiku.ainvoke(...)` / `await sonnet.ainvoke(...)` — the **async** (awaitable) twin of `.invoke(...)`. Every LangChain runnable (the chat clients *and* the compiled graph) offers both; the course defaults to the async one, so model-calling nodes are `async def` and runs are `await app.ainvoke(...)` (a notebook cell can `await` at top level). *Why* async is the default is the K05 prework's "why async for agents."

In [ ]:
class RouteState(TypedDict):
    """State for the routing workflow.

    `category` is set by the `classify` node (a model label); `user_choice` is set by the
    USER in the initial state (used in section 6). One state type carries both deciders.
    """

    ticket: str  # the raw ticket (input)
    category: str  # the classifier's label: "billing" | "technical" | "general"
    user_choice: str  # a path the user supplies directly (section 6)
    draft: str  # the chosen branch's reply
    steps: Annotated[list[str], add]


async def classify(state: RouteState) -> dict[str, object]:
    """Label the ticket. A LIGHT step -> Haiku (we only need one word)."""
    reply = await haiku.ainvoke(
        "Classify this support ticket as exactly one word -- billing, technical, or general.\n\n"
        f"{state['ticket']}"
    )
    label = str(reply.content).strip().lower()
    # Keep only a known label; if the model adds extra words, fall back to "general".
    category = next((c for c in ("billing", "technical", "general") if c in label), "general")
    return {"category": category, "steps": [f"classify->{category}"]}


def make_branch(name: str, instruction: str):
    """Build a branch node that drafts a reply with Sonnet. The model works INSIDE the node."""

    async def branch(state: RouteState) -> dict[str, object]:
        reply = await sonnet.ainvoke(f"{instruction}\n\nTicket: {state['ticket']}")
        return {"draft": str(reply.content), "steps": [name]}

    return branch


billing = make_branch(
    "billing", "Write a billing-support reply. Offer a refund only if policy allows."
)
technical = make_branch(
    "technical", "Write a technical-support reply. Ask for repro steps if useful."
)
general = make_branch("general", "Write a friendly general-support reply.")

[↑ Back to top](#top)

## 3. The conditional edge: branch on a label in state

`add_conditional_edges` takes a **routing function** that reads state and returns the *name* of the
next node. Here it returns `state["category"]` -- **a label your code already put in state.**

Say the line out loud: **this is not the model deciding the path.** The model produced a *label*;
your routing function read that label and chose the edge. In L11 the routing function will branch on
whether the *model asked for a tool* -- same mechanism, different decider.

In [ ]:
def route(state: RouteState) -> str:
    """Pick the branch from the label ALREADY in state -- no model call in this decision."""
    return state["category"]


builder = StateGraph(RouteState)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_node("general", general)
builder.set_entry_point("classify")
builder.add_conditional_edges(
    "classify",
    route,
    {"billing": "billing", "technical": "technical", "general": "general"},
)
for branch_name in ("billing", "technical", "general"):
    builder.add_edge(branch_name, END)
route_app = builder.compile()

# The diagram now visibly BRANCHES (dashed conditional edges) yet still always reaches END.
print(route_app.get_graph().draw_mermaid())

[↑ Back to top](#top)

## 4. Run it -- and prove it's deterministic

Invoke on one ticket per category, then run the *same* ticket twice and show the path is identical.
The branch wording may differ run to run; the **path** is stable, because you wired it.

In [ ]:
if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping the live run. Set it to watch routing execute.")
else:
    for kind in ("billing", "technical", "general"):
        out = await route_app.ainvoke(
            {"ticket": TICKETS[kind], "steps": []},
            config={"callbacks": trace_callbacks()},
        )
        print(f"{kind:9s} -> path {out['steps']}")

    # Determinism: same input, same path, twice.
    path_a = (await route_app.ainvoke({"ticket": TICKETS["billing"], "steps": []}))["steps"]
    path_b = (await route_app.ainvoke({"ticket": TICKETS["billing"], "steps": []}))["steps"]
    print("\nsame input, same path:", path_a == path_b, path_a)

[↑ Back to top](#top)

## 5. Read the trace (a forward-pointing taste, not a prerequisite) -- a model per node

If Langfuse is configured, the routing run shows only the **one chosen branch** -- not all three --
confirming the path. And the models differ per span: the `classify` span ran **Haiku** (cheap,
fast, just a label), the branch span ran **Sonnet** (the quality reply). Look for the two models on
two spans. (You'll get the full trace-reading skill in **L12**; here it's just a taste, same as L03.)

That's the *mechanism* of mixed-model design, reused from L03. The *which-model* decision
framework -- capability vs. latency vs. cost, budgets -- is **L14's** job (Choosing model power).
Here you're seeing only *that* you can mix, and *how*: each node constructed its own
`ChatAnthropic(model=...)`. If Langfuse isn't configured the run still works -- you just won't see
the spans.

[↑ Back to top](#top)

## 6. Same graph, different decider: user input

Now swap the **decider**. Instead of a `classify` node, the path comes from a `user_choice` value
supplied in the **initial state** -- a menu choice the user already made. The routing function reads
it directly: **no model in the routing decision at all.** This is the purest "you wire the flow".

Note the combined shape: an `intake` pass-through node (plain Python) routes on the user's choice,
then a **Sonnet branch node still drafts the reply** -- *the user owns the edge, the model works
inside the node.* To make it unmistakable, you'll feed a *technical* ticket but the user picks
`billing`: the path follows the **user**, not the ticket's content.

In [ ]:
def intake(state: RouteState) -> dict[str, object]:
    """A plain pass-through node -- no model. The user has already chosen the path."""
    return {"steps": ["intake"]}


def route_by_user(state: RouteState) -> str:
    """Branch on the USER's choice from the initial state. No model in this decision."""
    return state["user_choice"]


ub = StateGraph(RouteState)
ub.add_node("intake", intake)
ub.add_node("billing", billing)
ub.add_node("technical", technical)
ub.add_node("general", general)
ub.set_entry_point("intake")
ub.add_conditional_edges(
    "intake",
    route_by_user,
    {"billing": "billing", "technical": "technical", "general": "general"},
)
for branch_name in ("billing", "technical", "general"):
    ub.add_edge(branch_name, END)
user_app = ub.compile()

if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping the live run.")
else:
    # Technical ticket, but the USER chose 'billing' -> the path follows the user, not the content.
    out = await user_app.ainvoke(
        {"ticket": TICKETS["technical"], "user_choice": "billing", "steps": []}
    )
    print("user chose billing -> path", out["steps"])

[↑ Back to top](#top)

## 7. Takeaways

- **A conditional edge is not the model deciding.** Here the routing function branches on **state
  you set** -- a model *label* (section 3) or **direct user input** (section 6). In L11 it
  branches on whether the model asked for a tool. Call out which it is every time.
- **Routing can be decided by data, a model label, or the user.** User input is the sharpest
  contrast with an agent: here *you (or the user)* pick the path; in L11 the *model* does. Most
  real "AI workflows" mix the two.
- **Each node can use its own model.** Haiku where you need a label, Sonnet where you need quality
  -- reusing L03's mixed-model mechanism. The trace shows each span's model and cost (the *when/why*
  is L14).
- **Still a workflow either way.** You wired every branch; re-running takes the same path.
  The *interactive* "pause and ask the user" variant needs a checkpointer and is **L17**.

[↑ Back to top](#top)